# Extraction des données pour la fusion

In [ ]:
import os
import requests
import pandas as pd
import gzip
import time

## Réccupération automatique des données Météo,

Le problème de météo france est le fait qu'a l'heure actuel on doit téléchargé les mois 1 par 1.

In [4]:
output_folder = "climat_mensuel"
os.makedirs(output_folder, exist_ok=True)

start_year = 1990
end_year = 2025

base_url = "https://donneespubliques.meteofrance.fr/donnees_libres/Txt/Climat/climat.{date}.csv.gz"

for year in range(start_year, end_year + 1):
    for month in range(1, 13):
        date_str = f"{year}{month:02d}"
        url = base_url.format(date=date_str)
        local_file = os.path.join(output_folder, f"climat.{date_str}.csv.gz")

        if os.path.exists(local_file):
            print(f"[SKIP] {local_file} existe déjà")
            continue

        try:
            r = requests.get(url, timeout=10)
            if r.status_code == 200:
                with open(local_file, "wb") as f:
                    f.write(r.content)
                print(f"[OK] Téléchargé {date_str}")
            else:
                print(f"[MANQUANT] {date_str} (HTTP {r.status_code})")
        except Exception as e:
            print(f"[ERREUR] {date_str} : {e}")

print("Téléchargement terminé !")

[SKIP] climat_mensuel\climat.199001.csv.gz existe déjà
[SKIP] climat_mensuel\climat.199002.csv.gz existe déjà
[SKIP] climat_mensuel\climat.199003.csv.gz existe déjà
[SKIP] climat_mensuel\climat.199004.csv.gz existe déjà
[SKIP] climat_mensuel\climat.199005.csv.gz existe déjà
[SKIP] climat_mensuel\climat.199006.csv.gz existe déjà
[SKIP] climat_mensuel\climat.199007.csv.gz existe déjà
[SKIP] climat_mensuel\climat.199008.csv.gz existe déjà
[SKIP] climat_mensuel\climat.199009.csv.gz existe déjà
[SKIP] climat_mensuel\climat.199010.csv.gz existe déjà
[SKIP] climat_mensuel\climat.199011.csv.gz existe déjà
[SKIP] climat_mensuel\climat.199012.csv.gz existe déjà
[SKIP] climat_mensuel\climat.199101.csv.gz existe déjà
[SKIP] climat_mensuel\climat.199102.csv.gz existe déjà
[SKIP] climat_mensuel\climat.199103.csv.gz existe déjà
[SKIP] climat_mensuel\climat.199104.csv.gz existe déjà
[SKIP] climat_mensuel\climat.199105.csv.gz existe déjà
[SKIP] climat_mensuel\climat.199106.csv.gz existe déjà
[SKIP] cli

In [5]:
input_folder = "climat_mensuel"
output_file = "../data/in/meteo.csv"

all_files = [f for f in os.listdir(input_folder) if f.endswith(".csv.gz")]
all_files.sort()
dfs = []

for file in all_files:
    file_path = os.path.join(input_folder, file)
    
    with gzip.open(file_path, 'rt', encoding='utf-8') as f:
        df = pd.read_csv(f, sep=";")
        dfs.append(df)

meteo_complet = pd.concat(dfs, ignore_index=True)

meteo_complet.to_csv(output_file, sep=";", index=False)

print(f"[OK] {len(all_files)} fichiers fusionnés dans {output_file}")

[OK] 432 fichiers fusionnés dans ../data/in/meteo.csv


## Réccupération automatique des fichiers du niveau de l'eau des nappes

In [ ]:
import os
import requests
import pandas as pd
import time

# =========================
# Paramètres
# =========================
departement = "45"
output_folder = "nappe_piezometres"
os.makedirs(output_folder, exist_ok=True)

# =========================
# 1. Récupérer toutes les stations (pagination)
# =========================
url_stations = "https://hubeau.eaufrance.fr/api/v1/niveaux_nappes/stations"

codes = []
page = 1

while True:
    params = {
        "code_departement": departement,
        "size": 200,
        "page": page
    }
    
    r = requests.get(url_stations, params=params)
    data = r.json()["data"]
    
    if not data:
        break
    
    codes.extend([s["code_bss"] for s in data])
    print(f"Page {page} : {len(data)} stations")
    page += 1

codes = list(set(codes))
print(f"Total stations trouvées : {len(codes)}")

# =========================
# 2. Télécharger un CSV par piézomètre
# =========================
url_chroniques = "https://hubeau.eaufrance.fr/api/v1/niveaux_nappes/chroniques"

for i, code in enumerate(codes):
    params = {
        "code_bss": code,
        "size": 20000
    }
    
    try:
        r = requests.get(url_chroniques, params=params)
        data = r.json()["data"]
        
        if not data:
            print(f"[{i+1}/{len(codes)}] {code} : aucune donnée")
            continue
        
        df = pd.DataFrame(data)
        
        # Nom de fichier propre
        safe_code = code.replace("/", "_")
        file_path = os.path.join(output_folder, f"nappe_{safe_code}.csv")
        
        df.to_csv(file_path, sep=";", index=False)
        print(f"[{i+1}/{len(codes)}] {code} : {len(df)} lignes")
        
        time.sleep(0.2)  # éviter surcharge API
        
    except Exception as e:
        print(f"[{i+1}/{len(codes)}] Erreur {code} : {e}")

print("Téléchargement terminé")

Page 1 : 120 stations
Total stations trouvées : 120
[1/120] 03652X0055/S1 : 472 lignes
[2/120] 03283X0060/F : 248 lignes
[3/120] 03991X0039/P1 : 130 lignes
[4/120] 04324X0011/F : 2382 lignes
[5/120] BSS004NBLL/X : 380 lignes
[6/120] 04003X0018/FAEP2 : 10542 lignes
[7/120] 03624X0060/S1 : 3210 lignes
[8/120] 03278X0020/P : 9300 lignes
[9/120] 03634X0027/S1 : 1252 lignes
[10/120] 03273X0032/S1 : 326 lignes
[11/120] 03622X0023/S1 : 274 lignes
[12/120] 03637X0122/P : 1583 lignes
[13/120] 02928X1008/P : 1671 lignes
[14/120] 03646X0083/F : 534 lignes
[15/120] 03281X0019/P : 9240 lignes
[16/120] 03287X0018/S1 : 12527 lignes
[17/120] 03288X0096/F : 247 lignes
[18/120] 03636X0284/P1 : 108 lignes
[19/120] 03296X1010/P : 322 lignes
[20/120] 03277X0059/S1 : 392 lignes
[21/120] 03986X0028/P1 : 302 lignes
[22/120] 03991X0194/SZAP17 : 3353 lignes
[23/120] 03646X0086/F1 : 8702 lignes
[24/120] 03651X0107/P : 10749 lignes
[25/120] 04326X0034/F : 10964 lignes
[26/120] 03982X1045/F : 4949 lignes
[27/120] 

## Réccupération automatique des fichiers d'ETP

In [7]:
import requests
import os

dataset_id = "667eae35510cd549fc7722c1"
output_folder = "etp_fao_hargreaves"
os.makedirs(output_folder, exist_ok=True)

# Récupérer la liste des ressources
resp = requests.get(f"https://www.data.gouv.fr/api/1/datasets/{dataset_id}/")
data = resp.json()

for resource in data.get("resources", []):
    url = resource.get("url")
    fmt = resource.get("format", "").lower()
    if fmt in ["csv", "csv.gz"]:
        filename = os.path.join(output_folder, url.split("/")[-1])
        print(f"[DOWNLOAD] {url}")
        r = requests.get(url)
        with open(filename, "wb") as f:
            f.write(r.content)

print("Téléchargement ETP FAO Hargreaves terminé !")

[DOWNLOAD] https://object.files.data.gouv.fr/meteofrance/data/synchro_ftp/REF_CC/ETP_FAO/ETP_Hargreaves_coefficient_0.175_1970-1979.csv.gz
[DOWNLOAD] https://object.files.data.gouv.fr/meteofrance/data/synchro_ftp/REF_CC/ETP_FAO/ETP_Hargreaves_coefficient_0.175_1980-1989.csv.gz
[DOWNLOAD] https://object.files.data.gouv.fr/meteofrance/data/synchro_ftp/REF_CC/ETP_FAO/ETP_Hargreaves_coefficient_0.175_1990-1999.csv.gz
[DOWNLOAD] https://object.files.data.gouv.fr/meteofrance/data/synchro_ftp/REF_CC/ETP_FAO/ETP_Hargreaves_coefficient_0.175_2000-2009.csv.gz
[DOWNLOAD] https://object.files.data.gouv.fr/meteofrance/data/synchro_ftp/REF_CC/ETP_FAO/ETP_Hargreaves_coefficient_0.175_2010-2019.csv.gz


KeyboardInterrupt: 

## Réccupération automatique des fichiers lien entre etp et géoloc